In [1]:
# Phase 2 – Data Engineering
# Cell 1: Setup + Load Phase 1 data

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# ---------------------- Paths ----------------------
PROJECT = Path(r"D:\univercity\omid_project\economics_project")

PHASE1_DATA = PROJECT / "outputs" / "phase1" / "data"
OUT_DIR     = PROJECT / "outputs" / "phase2"

OUT_DATA  = OUT_DIR / "data"
OUT_TAB   = OUT_DIR / "tables"
OUT_FIG   = OUT_DIR / "figures"
OUT_LOG   = OUT_DIR / "logs"

for d in [OUT_DATA, OUT_TAB, OUT_FIG, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 1")
print("=" * 60)
print(f"Timestamp     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root  : {PROJECT}")
print(f"Outputs       : {OUT_DIR}")
print()

# ---------------------- Load Phase 1 data ----------------------
df_jst = pd.read_pickle(PHASE1_DATA / "jst_raw.pkl")
df_imf = pd.read_pickle(PHASE1_DATA / "imf_weo_raw.pkl")

print("Loaded from Phase 1:")
print(f"  JST  : {df_jst.shape[0]} rows × {df_jst.shape[1]} columns")
print(f"  IMF  : {df_imf.shape[0]} rows × {df_imf.shape[1]} columns")
print("\nCell 1 completed successfully.")

Phase 2 – Data Engineering | Cell 1
Timestamp     : 2026-09-22 09:55:33
Project Root  : D:\univercity\omid_project\economics_project
Outputs       : D:\univercity\omid_project\economics_project\outputs\phase2

Loaded from Phase 1:
  JST  : 2718 rows × 59 columns
  IMF  : 8200 rows × 117 columns

Cell 1 completed successfully.


In [2]:
# Phase 2 – Data Engineering
# Cell 2: Data Cleaning + Missing Values Analysis (JST)

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

# Load
df_jst = pd.read_pickle(PROJECT / "outputs" / "phase1" / "data" / "jst_raw.pkl")

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 2")
print("JST Cleaning + Missing Values Analysis")
print("=" * 60)

# ---------------------- Basic Cleaning ----------------------
print("\n--- Basic Info before cleaning ---")
print(f"Shape: {df_jst.shape}")
print(f"Duplicated rows: {df_jst.duplicated().sum()}")

# Remove exact duplicates if any
df_jst = df_jst.drop_duplicates().copy()

# Ensure correct types
df_jst["year"] = df_jst["year"].astype(int)

print(f"Shape after removing duplicates: {df_jst.shape}")

# ---------------------- Missing Values Analysis ----------------------
print("\n--- Missing Values Summary ---")
missing = df_jst.isnull().sum()
missing_pct = (df_jst.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing_Count": missing.values,
    "Missing_Percent": missing_pct.values
}).sort_values("Missing_Percent", ascending=False)

print(missing_df.head(20).to_string(index=False))

# Save missing report
missing_df.to_csv(OUT_TAB / "jst_missing_report.csv", index=False)
print(f"\nSaved missing report → {OUT_TAB / 'jst_missing_report.csv'}")

# ---------------------- Columns with very high missing ----------------------
high_missing = missing_df[missing_df["Missing_Percent"] > 50]
print(f"\nColumns with >50% missing: {len(high_missing)}")
if len(high_missing) > 0:
    print(high_missing[["Column", "Missing_Percent"]].to_string(index=False))

# ---------------------- Save cleaned version (so far) ----------------------
df_jst.to_pickle(OUT_DATA / "jst_clean_step1.pkl")
print(f"\nSaved intermediate cleaned data → {OUT_DATA / 'jst_clean_step1.pkl'}")

print("\nCell 2 completed successfully.")

Phase 2 – Data Engineering | Cell 2
JST Cleaning + Missing Values Analysis

--- Basic Info before cleaning ---
Shape: (2718, 59)
Duplicated rows: 0
Shape after removing duplicates: (2718, 59)

--- Missing Values Summary ---
                  Column  Missing_Count  Missing_Percent
housing_capgain_ipolated           2713            99.82
            eq_dp_interp           2712            99.78
            eq_tr_interp           2711            99.74
       eq_capgain_interp           2711            99.74
           rent_ipolated           2689            98.93
                    tbus           1419            52.21
                     thh           1344            49.45
              capital_tr            888            32.67
                risky_tr            864            31.79
                   bdebt            853            31.38
        housing_rent_rtn            809            29.76
              housing_tr            809            29.76
         housing_rent_yd           

In [3]:
# Phase 2 – Data Engineering
# Cell 3: Handle Missing Values + Outlier Detection (JST)

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

df = pd.read_pickle(OUT_DATA / "jst_clean_step1.pkl")

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 3")
print("Missing Values Treatment + Outlier Detection")
print("=" * 60)

# ---------------------- 1. Drop columns with extreme missing (>90%) ----------------------
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 90].index.tolist()

print(f"\nColumns dropped (>90% missing): {len(cols_to_drop)}")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print(f"Shape after dropping high-missing columns: {df.shape}")

# ---------------------- 2. Core variables we want to keep ----------------------
core_vars = [
    "year", "country", "iso", "pop",
    "rgdpmad", "rgdpbarro", "gdp", "iy",
    "cpi", "stir", "ltrate", "money", "narrowm",
    "unemp", "wage", "debtgdp", "revenue", "expenditure",
    "tloans", "tmort", "thh", "tbus", "bdebt",
    "crisisJST", "ca", "imports", "exports"
]

# Keep only columns that exist
core_vars = [c for c in core_vars if c in df.columns]
df_core = df[core_vars].copy()

print(f"\nCore variables selected: {len(core_vars)}")
print(core_vars)

# ---------------------- 3. Outlier Detection (IQR method on numeric columns) ----------------------
numeric_cols = df_core.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["year", "crisisJST"]]

outlier_summary = []

for col in numeric_cols:
    Q1 = df_core[col].quantile(0.25)
    Q3 = df_core[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    n_outliers = ((df_core[col] < lower) | (df_core[col] > upper)).sum()
    outlier_summary.append({
        "Column": col,
        "Outliers_Count": n_outliers,
        "Outliers_Percent": round(n_outliers / len(df_core) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("Outliers_Percent", ascending=False)
print("\n--- Outlier Summary (IQR × 3) ---")
print(outlier_df.head(15).to_string(index=False))

outlier_df.to_csv(OUT_TAB / "jst_outlier_report.csv", index=False)
print(f"\nSaved outlier report → {OUT_TAB / 'jst_outlier_report.csv'}")

# ---------------------- 4. Save current state ----------------------
df_core.to_pickle(OUT_DATA / "jst_core_clean.pkl")
print(f"Saved core cleaned data → {OUT_DATA / 'jst_core_clean.pkl'}")

print("\nCell 3 completed successfully.")

Phase 2 – Data Engineering | Cell 3
Missing Values Treatment + Outlier Detection

Columns dropped (>90% missing): 5
['rent_ipolated', 'housing_capgain_ipolated', 'eq_capgain_interp', 'eq_tr_interp', 'eq_dp_interp']
Shape after dropping high-missing columns: (2718, 54)

Core variables selected: 27
['year', 'country', 'iso', 'pop', 'rgdpmad', 'rgdpbarro', 'gdp', 'iy', 'cpi', 'stir', 'ltrate', 'money', 'narrowm', 'unemp', 'wage', 'debtgdp', 'revenue', 'expenditure', 'tloans', 'tmort', 'thh', 'tbus', 'bdebt', 'crisisJST', 'ca', 'imports', 'exports']

--- Outlier Summary (IQR × 3) ---
     Column  Outliers_Count  Outliers_Percent
         ca             859             31.60
    exports             518             19.06
    imports             511             18.80
        gdp             504             18.54
     tloans             503             18.51
    revenue             499             18.36
      money             498             18.32
expenditure             498             18.32

In [5]:
# Phase 2 – Data Engineering
# Cell 4: Stationarity Tests (ADF + KPSS)

import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore")

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

df = pd.read_pickle(OUT_DATA / "jst_core_clean.pkl")

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 4")
print("Stationarity Tests (ADF + KPSS)")
print("=" * 60)

# Variables to test (important numeric ones)
vars_to_test = [
    "rgdpmad", "gdp", "cpi", "stir", "ltrate", "money",
    "unemp", "debtgdp", "revenue", "expenditure",
    "tloans", "iy", "ca"
]

vars_to_test = [v for v in vars_to_test if v in df.columns]

results = []

for country in sorted(df["country"].unique()):
    cdf = df[df["country"] == country].sort_values("year")
    
    for var in vars_to_test:
        series = cdf[var].dropna()
        
        if len(series) < 20:
            continue
        
        # ADF test
        try:
            adf_stat, adf_pval, *_ = adfuller(series, autolag="AIC")
        except:
            adf_stat, adf_pval = np.nan, np.nan
        
        # KPSS test
        try:
            kpss_stat, kpss_pval, *_ = kpss(series, regression="c", nlags="auto")
        except:
            kpss_stat, kpss_pval = np.nan, np.nan
        
        results.append({
            "country": country,
            "variable": var,
            "n_obs": len(series),
            "ADF_stat": round(adf_stat, 4) if not np.isnan(adf_stat) else np.nan,
            "ADF_pvalue": round(adf_pval, 4) if not np.isnan(adf_pval) else np.nan,
            "KPSS_stat": round(kpss_stat, 4) if not np.isnan(kpss_stat) else np.nan,
            "KPSS_pvalue": round(kpss_pval, 4) if not np.isnan(kpss_pval) else np.nan
        })

res_df = pd.DataFrame(results)

# Interpretation helper
res_df["ADF_stationary"] = res_df["ADF_pvalue"] < 0.05
res_df["KPSS_stationary"] = res_df["KPSS_pvalue"] > 0.05

print("\n--- Stationarity Results (sample) ---")
print(res_df.head(15).to_string(index=False))

# Summary by variable
summary = res_df.groupby("variable").agg(
    countries_tested=("country", "count"),
    ADF_stationary_pct=("ADF_stationary", "mean"),
    KPSS_stationary_pct=("KPSS_stationary", "mean")
).round(3)

print("\n--- Summary by Variable ---")
print(summary)

# Save
res_df.to_csv(OUT_TAB / "stationarity_tests_detailed.csv", index=False)
summary.to_csv(OUT_TAB / "stationarity_summary_by_variable.csv")
print(f"\nSaved detailed results → {OUT_TAB / 'stationarity_tests_detailed.csv'}")
print(f"Saved summary         → {OUT_TAB / 'stationarity_summary_by_variable.csv'}")

print("\nCell 4 completed successfully.")

Phase 2 – Data Engineering | Cell 4
Stationarity Tests (ADF + KPSS)

--- Stationarity Results (sample) ---
  country    variable  n_obs  ADF_stat  ADF_pvalue  KPSS_stat  KPSS_pvalue  ADF_stationary  KPSS_stationary
Australia     rgdpmad    151    2.6440      0.9991     1.5342       0.0100           False            False
Australia         gdp    151    1.2955      0.9966     1.0340       0.0100           False            False
Australia         cpi    151    0.4192      0.9822     1.2809       0.0100           False            False
Australia        stir    148   -1.8315      0.3649     0.3483       0.0994           False             True
Australia      ltrate    151   -2.4006      0.1416     0.4791       0.0464           False            False
Australia       money    151    4.5228      1.0000     0.8715       0.0100           False            False
Australia       unemp    120   -3.2327      0.0182     0.1107       0.1000            True             True
Australia     debtgdp    151 

In [6]:
# Phase 2 – Data Engineering
# Cell 5: Create Differenced / Growth versions + Final Clean Panel

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

df = pd.read_pickle(OUT_DATA / "jst_core_clean.pkl")

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 5")
print("Creating Differenced Series + Final Panel")
print("=" * 60)

# Sort
df = df.sort_values(["country", "year"]).reset_index(drop=True)

# Variables to difference / create growth rates
level_vars = [
    "rgdpmad", "gdp", "cpi", "money", "narrowm",
    "debtgdp", "revenue", "expenditure",
    "tloans", "tmort", "thh", "tbus", "bdebt",
    "pop", "wage", "imports", "exports"
]
level_vars = [v for v in level_vars if v in df.columns]

# Create first difference and log-difference (growth)
for var in level_vars:
    df[f"{var}_diff"] = df.groupby("country")[var].diff()
    # Log growth where positive values exist
    df[f"{var}_logdiff"] = df.groupby("country")[var].transform(
        lambda x: np.log(x.replace(0, np.nan)).diff()
    )

print(f"\nAdded differenced and log-differenced columns for {len(level_vars)} variables.")
print(f"New shape: {df.shape}")

# Keep interest rates and ratios as is (usually more stationary)
# crisisJST remains binary

# Final column order
id_cols = ["year", "country", "iso"]
core_level = [c for c in df.columns if c in level_vars or c in ["stir", "ltrate", "unemp", "iy", "ca", "crisisJST"]]
diff_cols = [c for c in df.columns if c.endswith("_diff") or c.endswith("_logdiff")]

final_cols = id_cols + core_level + diff_cols
final_cols = list(dict.fromkeys(final_cols))  # remove duplicates while preserving order

df_final = df[final_cols].copy()

# Save final cleaned panel
df_final.to_pickle(OUT_DATA / "jst_final_clean.pkl")
df_final.to_csv(OUT_DATA / "jst_final_clean.csv", index=False)

print(f"\nFinal clean panel saved:")
print(f"  → {OUT_DATA / 'jst_final_clean.pkl'}")
print(f"  → {OUT_DATA / 'jst_final_clean.csv'}")
print(f"Final shape: {df_final.shape}")

# Quick missing summary of final data
miss_final = df_final.isnull().mean().sort_values(ascending=False).head(10)
print("\nTop missing in final panel:")
print(miss_final.round(3))

print("\nCell 5 completed successfully.")

Phase 2 – Data Engineering | Cell 5
Creating Differenced Series + Final Panel

Added differenced and log-differenced columns for 17 variables.
New shape: (2718, 61)

Final clean panel saved:
  → D:\univercity\omid_project\economics_project\outputs\phase2\data\jst_final_clean.pkl
  → D:\univercity\omid_project\economics_project\outputs\phase2\data\jst_final_clean.csv
Final shape: (2718, 60)

Top missing in final panel:
tbus_diff        0.529
tbus_logdiff     0.529
tbus             0.522
thh_logdiff      0.501
thh_diff         0.501
thh              0.494
bdebt_diff       0.323
bdebt_logdiff    0.323
bdebt            0.314
unemp            0.283
dtype: float64

Cell 5 completed successfully.
